# 15.4 Learning to rank

A ranking model should be trained on the kind of mistake it will be judged for: bad scores, bad pairs, or bad lists.

This lesson moves from prediction to ordering. Pointwise, pairwise, listwise, and LambdaRank-style NDCG-weighted pair losses exert different pressure, so the notebook compares them on the same slates through NDCG@3. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1500
rng = np.random.default_rng(SEED)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def cosine(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0.0:
        return 0.0
    return float(np.dot(a, b) / denom)


def ndcg_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    gains = (2.0 ** relevance[order] - 1.0) / np.log2(np.arange(2, len(order) + 2))
    ideal = np.argsort(-relevance, kind="mergesort")[:k]
    ideal_gains = (2.0 ** relevance[ideal] - 1.0) / np.log2(np.arange(2, len(ideal) + 2))
    denom = np.sum(ideal_gains)
    if denom == 0.0:
        return 0.0
    return float(np.sum(gains) / denom)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def recall_at_k(relevance, scores, k=3):
    relevance = np.asarray(relevance, dtype=float) > 0.0
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores, kind="mergesort")[:k]
    denom = np.sum(relevance)
    if denom == 0:
        return 0.0
    return float(np.sum(relevance[order]) / denom)


def rating_ladder():
    d1 = np.array([
        [5.0, 4.0, 0.0, 1.0],
        [4.0, 5.0, 0.0, 1.0],
        [1.0, 1.0, 5.0, 4.0],
        [0.0, 1.0, 4.0, 5.0],
    ])
    d2 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0],
        [4.0, 5.0, 0.0, 1.0, 2.0],
        [1.0, 1.0, 5.0, 4.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0],
        [5.0, 0.0, 1.0, 0.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0],
    ])
    d3 = np.array([
        [5.0, 4.0, 0.0, 1.0, 0.0, 2.0],
        [4.0, 4.0, 0.0, 0.0, 2.0, 0.0],
        [1.0, 1.0, 5.0, 4.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 5.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 4.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 3.0, 0.0, 3.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 1.0],
    ])
    d4 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0],
        [4.0, 5.0, 0.0, 2.0, 0.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [1.0, 0.0, 4.0, 5.0, 0.0, 5.0, 0.0, 2.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0],
        [0.0, 3.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 1.0, 4.0, 0.0],
        [0.0, 4.0, 0.0, 5.0, 0.0, 5.0, 0.0, 3.0],
        [4.0, 0.0, 2.0, 0.0, 5.0, 0.0, 4.0, 1.0],
        [0.0, 2.0, 4.0, 5.0, 0.0, 4.0, 0.0, 0.0],
    ])
    d5 = np.array([
        [5.0, 4.0, 0.0, 0.0, 1.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [4.0, 5.0, 0.0, 0.0, 1.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 5.0, 4.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 4.0, 5.0, 0.0, 5.0, 0.0, 0.0, 0.0, 2.0],
        [0.0, 0.0, 5.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, 0.0],
        [5.0, 0.0, 1.0, 0.0, 4.0, 0.0, 5.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 1.0, 0.0, 0.0],
        [0.0, 4.0, 0.0, 4.0, 0.0, 5.0, 0.0, 3.0, 0.0, 0.0],
        [3.0, 0.0, 0.0, 0.0, 5.0, 0.0, 4.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 4.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 5.0],
    ])
    return [
        {"name": "D1 tiny rating matrix", "ratings": d1, "holdout": [(0, 2, 5.0)]},
        {"name": "D2 synthetic preferences", "ratings": d2, "holdout": [(0, 2, 5.0), (2, 4, 2.0), (5, 4, 3.0)]},
        {"name": "D3 noise ties sparsity", "ratings": d3, "holdout": [(0, 2, 5.0), (1, 3, 2.0), (6, 1, 3.0), (7, 4, 2.0)]},
        {"name": "D4 inline MovieLens-like sample", "ratings": d4, "holdout": [(0, 2, 4.0), (1, 5, 3.0), (4, 3, 2.0), (6, 7, 2.0)]},
        {"name": "D5 long-tail cold-start", "ratings": d5, "holdout": [(0, 2, 4.0), (2, 9, 2.0), (5, 8, 4.0), (10, 0, 3.0), (11, 1, 3.0)]},
    ]


def slate_ladder():
    return [
        {"name": "D1 3-item slate", "features": np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.744, 0.643, 0.819]), "collab": np.array([0.30, 0.90, 0.70]), "relevance": np.array([2.0, 3.0, 1.0])},
        {"name": "D2 synthetic preferences", "features": np.array([[1.0, 0.1, 0.0], [0.2, 0.9, 0.8], [0.8, 0.7, 0.1], [0.1, 0.2, 1.0]]), "content": np.array([0.70, 0.62, 0.88, 0.55]), "collab": np.array([0.40, 0.85, 0.65, 0.30]), "relevance": np.array([1.0, 3.0, 2.0, 0.0])},
        {"name": "D3 noise ties sparsity", "features": np.array([[1.0, 0.0, 0.2], [0.9, 0.2, 0.0], [0.1, 1.0, 0.9], [0.2, 0.8, 0.8], [0.0, 0.1, 1.0]]), "content": np.array([0.78, 0.78, 0.59, 0.60, 0.51]), "collab": np.array([0.20, 0.50, 0.88, 0.40, 0.10]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 0.0])},
        {"name": "D4 inline MovieLens-like ratings", "features": np.array([[1.0, 0.3, 0.0], [0.9, 0.4, 0.1], [0.2, 0.9, 0.6], [0.1, 0.6, 1.0], [0.7, 0.2, 0.8], [0.0, 0.1, 0.9]]), "content": np.array([0.82, 0.80, 0.66, 0.59, 0.75, 0.50]), "collab": np.array([0.55, 0.61, 0.92, 0.72, 0.35, 0.25]), "relevance": np.array([2.0, 2.0, 3.0, 1.0, 2.0, 0.0])},
        {"name": "D5 long-tail cold-start", "features": np.array([[1.0, 0.2, 0.0], [0.8, 0.1, 0.1], [0.1, 0.9, 0.8], [0.0, 0.7, 1.0], [0.9, 0.9, 0.0], [0.1, 0.0, 1.0], [1.0, 1.0, 0.0]]), "content": np.array([0.84, 0.78, 0.61, 0.58, 0.90, 0.52, 0.82]), "collab": np.array([0.65, 0.20, 0.91, 0.55, np.nan, 0.05, np.nan]), "relevance": np.array([2.0, 1.0, 3.0, 2.0, 3.0, 0.0, 2.0])},
    ]


def print_ladder_preview(rungs):
    for rung in rungs:
        if "ratings" in rung:
            matrix = rung["ratings"]
            observed = int(np.sum(matrix > 0.0))
            total = int(matrix.size)
            print(rung["name"], matrix.shape, "observed", observed, "of", total)
            print(matrix[:3, :5])
        else:
            relevance = rung["relevance"]
            print(rung["name"], "items", len(relevance), "positives", int(np.sum(relevance > 0.0)))
            print("relevance", relevance[:5])


## The concept, built once: pointwise, pairwise, listwise

Pointwise logistic loss treats items independently. Pairwise losses use $\max(0,1-(s_+-s_-))$ or $\log(1+e^{-(s_+-s_-)})$. LambdaRank weights pair gradients by the NDCG change from swapping two ranks. Listwise softmax uses $$L_{list}=-\lograc{e^{s_t}}{\sum_j e^{s_j}}.$$

In [ ]:

def rank_loss_scores(scores, labels, target=0):
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=float)
    probs = sigmoid(scores)
    pointwise = -labels * np.log(probs) - (1.0 - labels) * np.log(1.0 - probs)
    positives = np.where(labels > 0.0)[0]
    negatives = np.where(labels == 0.0)[0]
    hinge_terms = []
    logistic_terms = []
    for pos in positives:
        for neg in negatives:
            diff = scores[pos] - scores[neg]
            hinge_terms.append(max(0.0, 1.0 - diff))
            logistic_terms.append(np.log1p(np.exp(-diff)))
    shifted = scores - np.max(scores)
    softmax = np.exp(shifted) / np.sum(np.exp(shifted))
    listwise = -np.log(softmax[target])
    gradient = softmax.copy()
    gradient[target] -= 1.0
    return {
        "pointwise": float(np.mean(pointwise)),
        "pairwise_hinge": float(np.mean(hinge_terms)),
        "pairwise_logistic": float(np.mean(logistic_terms)),
        "listwise": float(listwise),
        "softmax": softmax,
        "gradient": gradient,
    }


def lambda_rank_grad(scores, relevance, k=3):
    scores = np.asarray(scores, dtype=float)
    relevance = np.asarray(relevance, dtype=float)
    order = np.argsort(-scores, kind="mergesort")
    ranks = np.empty_like(order)
    ranks[order] = np.arange(len(order))
    ideal_order = np.argsort(-relevance, kind="mergesort")
    ideal_gains = 2.0 ** relevance[ideal_order[:k]] - 1.0
    ideal_discounts = 1.0 / np.log2(np.arange(2, len(ideal_gains) + 2))
    idcg = float(np.sum(ideal_gains * ideal_discounts))
    if idcg == 0.0:
        return np.zeros_like(scores)
    grad = np.zeros_like(scores)
    for i in range(len(scores)):
        for j in range(len(scores)):
            if relevance[i] <= relevance[j]:
                continue
            rank_i = ranks[i]
            rank_j = ranks[j]
            if rank_i >= k and rank_j >= k:
                continue
            gain_i = 2.0 ** relevance[i] - 1.0
            gain_j = 2.0 ** relevance[j] - 1.0
            discount_i = 1.0 / np.log2(rank_i + 2.0)
            discount_j = 1.0 / np.log2(rank_j + 2.0)
            delta = abs((gain_i - gain_j) * (discount_i - discount_j)) / idcg
            rho = 1.0 / (1.0 + np.exp(scores[i] - scores[j]))
            lam = rho * delta
            grad[i] -= lam
            grad[j] += lam
    return grad


def train_linear_ranker(features, relevance, loss="listwise", steps=120, eta=0.05):
    features = np.asarray(features, dtype=float)
    relevance = np.asarray(relevance, dtype=float)
    w = np.zeros(features.shape[1])
    labels = (relevance == np.max(relevance)).astype(float)
    for step in range(steps):
        scores = features @ w
        if loss == "pointwise":
            grad_scores = sigmoid(scores) - labels
        elif loss == "pairwise":
            grad_scores = np.zeros_like(scores)
            positives = np.where(labels > 0.0)[0]
            negatives = np.where(labels == 0.0)[0]
            for pos in positives:
                for neg in negatives:
                    diff = scores[pos] - scores[neg]
                    weight = 1.0 / (1.0 + np.exp(diff))
                    grad_scores[pos] -= weight
                    grad_scores[neg] += weight
        elif loss == "lambda":
            grad_scores = lambda_rank_grad(scores, relevance, k=3)
        else:
            target = int(np.argmax(relevance))
            shifted = scores - np.max(scores)
            probs = np.exp(shifted) / np.sum(np.exp(shifted))
            grad_scores = probs
            grad_scores[target] -= 1.0
        grad_w = features.T @ grad_scores / len(scores)
        w = w - eta * grad_w
    return w


## Check the lesson numbers

For labels $(1,0,1)$ and scores $(2.0,0.5,1.0)$, pointwise loss is $0.471$. For a $0.500$ score gap, hinge loss is $0.500$ and smooth pairwise loss is $0.474$. Listwise target probability is $0.629$, loss is $0.464$, and the gradient is $(-0.371,0.140,0.231)$.

In [ ]:

losses = rank_loss_scores(np.array([2.0, 0.5, 1.0]), np.array([1.0, 0.0, 1.0]), target=0)
gap = 1.2 - 0.7
hinge = max(0.0, 1.0 - gap)
smooth = np.log1p(np.exp(-gap))

assert np.round(losses["pointwise"], 3) == 0.471
assert np.round(hinge, 3) == 0.500
assert np.round(smooth, 3) == 0.474
assert np.round(losses["softmax"][0], 3) == 0.629
assert np.round(losses["listwise"], 3) == 0.464
assert np.allclose(np.round(losses["gradient"], 3), np.array([-0.371, 0.140, 0.231]))

print("pointwise", round(losses["pointwise"], 3))
print("pairwise", round(hinge, 3), round(smooth, 3))
print("listwise", round(losses["listwise"], 3))


## The dataset ladder

The inline ladder reuses slates with content/collab features as ranker inputs and graded relevance as the query-local target.

In [ ]:

rungs = slate_ladder()
print_ladder_preview(rungs)


In [ ]:

rows = []
for rung in rungs:
    features = np.column_stack([rung["content"], np.nan_to_num(rung["collab"], nan=0.0), np.ones(len(rung["relevance"]))])
    w = train_linear_ranker(features, rung["relevance"], loss="lambda", steps=160, eta=0.08)
    scores = features @ w
    metric = ndcg_at_k(rung["relevance"], scores, k=3)
    rows.append({"name": rung["name"], "scores": scores, "metric": metric, "relevance": rung["relevance"]})

for row in rows:
    print(f"{row['name']}: NDCG@3={row['metric']:.3f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
last = rows[-1]
order = np.argsort(-last["scores"])
axes[0].bar(np.arange(len(order)), last["relevance"][order], color="darkorange")
axes[0].set_title("D5 relevance in learned rank order")
axes[0].set_xlabel("rank")
axes[0].set_ylabel("graded relevance")
axes[1].plot([row["name"].split()[0] for row in rows], [row["metric"] for row in rows], marker="o")
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title("NDCG@3 vs sparsity rung")
axes[1].set_ylabel("NDCG@3")
fig.tight_layout()


## Pitfall on D5: pointwise mismatch and easy negatives

Optimizing independent click labels can leave hard top-of-slate swaps unfixed. Pairwise/listwise training within each query focuses on the ordering metric.

In [ ]:

d5 = slate_ladder()[-1]
features = np.column_stack([d5["content"], np.nan_to_num(d5["collab"], nan=0.0), np.ones(len(d5["relevance"]))])
w_point = train_linear_ranker(features, d5["relevance"], loss="pointwise", steps=80, eta=0.05)
w_lambda = train_linear_ranker(features, d5["relevance"], loss="lambda", steps=160, eta=0.08)
point_metric = ndcg_at_k(d5["relevance"], features @ w_point, k=3)
lambda_metric = ndcg_at_k(d5["relevance"], features @ w_lambda, k=3)

print("pointwise NDCG@3", round(point_metric, 3))
print("LambdaRank NDCG@3", round(lambda_metric, 3))


## Evaluate it + Practice

Evaluation checklist:
- Track `NDCG@3` on D1--D5 and compare with a no-skill baseline such as popularity or original order.
- Run a sanity check where the strongest observed signal is swapped and the top item changes.
- Ablate the key idea, such as masking, latent factors, calibration, or query-local losses, and confirm the metric drops.
- Watch failure signals: identical recommendations for everyone, head-item dominance, unstable tiny-overlap scores, and poor cold-start behavior.

Practice:
1. Change one D3 tie and predict which slate position moves before running the cell.


2. Switch `loss` to `pairwise`, `listwise`, and `lambda` and compare hard-negative behavior.
3. Remove query-local normalization and explain why it is invalid.